# Phase 1 Starter — Train & Deploy (Individual, 35%)
This is Session 2's demo generalized: point it at **your** dataset and work top to bottom.
Milestones: S1 pick dataset → S2 first training run → S3 tuned + endpoint attempt →
**working endpoint by start of S4** → report (template in `project/`) due S5.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [ ]:
from course_utils import data
DATASET = 'bank_marketing'        # <- your pick from project/datasets.md
TARGET = 'y'                      # <- your label column
df = data.load_raw(DATASET)
df.head()

## 1. Profile & clean (Session 1 toolkit) — document ≥2 decisions for the report

In [ ]:
data.profile_frame(df)
# your cleaning here. Watch for target leakage: is any feature only known AFTER the outcome?


## 2. Features & splits

In [ ]:
import pandas as pd
y = (df[TARGET] == 'yes').astype(int) if df[TARGET].dtype == object else df[TARGET]
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)
S = data.make_splits(X, y)
print({k: v.shape for k, v in S.items() if hasattr(v, 'shape')})

## 3. Baseline → tuned model (≥2 hyperparameters, show the effect)

In [ ]:
from xgboost import XGBClassifier
from course_utils import viz
model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                      eval_metric='auc', random_state=42).fit(S['X_train'], S['y_train'])
m = viz.metric_report(S['y_val'], model.predict_proba(S['X_val'])[:, 1], name='baseline (val)')
# your tuning table here (see session2/tasks/task_hyperparameters.ipynb for the pattern)

## 4. Final honest number — test set, touched ONCE

In [ ]:
final = viz.metric_report(S['y_test'], model.predict_proba(S['X_test'])[:, 1], name='FINAL (test)')
from course_utils import models
models.save_model(model, f'phase1_{DATASET}', metrics=final)

## 5. Deploy
```python
from course_utils import aws
aws.deploy_sklearn_model(f'models/phase1_{DATASET}.joblib', endpoint_name=f'phase1-{DATASET}-YOURNAME')
# invoke it, screenshot it for the report, then DELETE it:
# aws.delete_endpoint(f'phase1-{DATASET}-YOURNAME')
```
Sandbox blocking you? Local FastAPI twin accepted with prior approval — see `session4/local_api.py`.